In [1]:
import pandas as pd

# Read data from Excel file
excel_file = 'icd10cm.xlsx'  # Update with your file path
df = pd.read_excel(excel_file)


In [3]:
import pandas as pd
import sqlite3
# Connect to phpmyadmin database codificacao
import mysql.connector as mysql

mysql_conn = mysql.connect(
    host="localhost",
    user="root",
    password="",
    database="codificacao"
)

#conn = sqlite3.connect('../database/database.sqlite')


df_transformado = df.rename(columns={
    'Código': 'codigo',
    'Descrição PT_(Longa)': 'descricao_longa',
    'Descrição PT_(Curta)': 'descricao_curta',
    'Capitulo ICD-10-CM_ Código': 'capitulo_codigo',
    'Capitulo ICD-10-CM_desc': 'capitulo_descricao',
    'Capitulo ICD-10-CM_desc_PT': 'capitulo_descricao_pt',
    'Secção ICD-10-CM_Código': 'secao_codigo',
    'Secção ICD-10-CM_Desc': 'secao_descricao',
    'Secção ICD-10-CM_Desc_PT': 'secao_descricao_pt',
    'Válido': 'valido',
    'Ano inicio ': 'ano_inicio',
    'Ano fim': 'ano_fim',
    'Versão': 'versao',
    'Codigo versão anterior': 'codigo_versao_anterior',
    'Tipo Alteração': 'tipo_alteracao'
})

# Selecionar somente as colunas desejadas
colunas_desejadas = [
        'codigo','descricao_longa','descricao_curta','valido','versao','codigo_versao_anterior','tipo_alteracao'
]

df_final = df_transformado[colunas_desejadas]

print(df_final.columns)

# conn.execute('DROP TABLE icd10cms;')
mysql_conn.cursor().execute('DROP TABLE IF EXISTS icd10cms_new;')

df_final.to_sql('icd10cms', mysql_conn, if_exists='replace', index=False)
# add primary key to the table
mysql_conn.cursor().execute('''
CREATE TABLE icd10cms_new (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    codigo TEXT,
    descricao_longa TEXT,
    descricao_curta TEXT,
    valido TEXT,
    versao TEXT,
    codigo_versao_anterior TEXT,
    tipo_alteracao TEXT
);
''')
mysql_conn.cursor().execute('''
INSERT INTO icd10cms_new (codigo, descricao_longa, descricao_curta, valido, versao, codigo_versao_anterior, tipo_alteracao)
SELECT codigo, descricao_longa, descricao_curta, valido, versao, codigo_versao_anterior, tipo_alteracao
FROM icd10cms;
''')

mysql_conn.cursor().execute('DROP TABLE icd10cms;')
mysql_conn.cursor().execute('ALTER TABLE icd10cms_new RENAME TO icd10cms;')

mysql_conn.commit()

# Close the database connection
print("Database connection closed.")

mysql_conn.close()

Index(['codigo', 'descricao_longa', 'descricao_curta', 'valido', 'versao',
       'codigo_versao_anterior', 'tipo_alteracao'],
      dtype='object')


C:\Users\pjcla\AppData\Local\Temp\ipykernel_36728\844576194.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_final.to_sql('icd10cms', mysql_conn, if_exists='replace', index=False)


DatabaseError: Execution failed on sql '
        SELECT
            name
        FROM
            sqlite_master
        WHERE
            type IN ('table', 'view')
            AND name=?;
        ': Not all parameters were used in the SQL statement